# 02 — Product Backorder Prediction

Portfolio edition of the original course analysis. Outputs are historical and were not rerun. See `reports/FINDINGS.md` for limitations. Code edits address imports, stale variable names and local data paths only.

Part 3 retrains on the saved LOF-filtered development subset, not the full smart sample.

In [77]:
from sklearn.model_selection import train_test_split
import os
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['NUMEXPR_NUM_THREADS'] = '2'

%matplotlib inline
import matplotlib.pyplot as plt

import os, sys
import itertools
import numpy as np
import pandas as pd
import joblib


## Reload the smart sample here

In [78]:

# Reload your smart sample from local file 
# ----------------------------------
sampled_X, sampled_y = joblib.load('sample-data.pkl')


## Split the data into Train/Test

Note that `train_test_split` does NOT do stratified sampling by default. If you have decided to have some imbalance in your sample, make sure to use the `stratify` option. 

In [79]:
X_train, X_test, y_train, y_test = train_test_split(sampled_X, sampled_y, test_size=0.2, stratify=sampled_y, random_state=42)

In [111]:
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest

from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2, f_classif, mutual_info_classif

from sklearn.pipeline import Pipeline
from time import time
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import LocalOutlierFactor
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.covariance import EllipticEnvelope
import pickle

---


### Your 1st pipeline 
  * Anomaly detection
  * Dimensionality reduction
  * Model training/validation
  
Add cells as needed. 

In [81]:
# ----------------------------------

#First pipeline: IsolationForest -> PCA -> Logistic Regression

iso_forest = IsolationForest(contamination=0.05).fit(X_train, y_train)
iso_outliers = iso_forest.predict(X_train) == -1
print(f"Num of outliers = {np.sum(iso_outliers)}")
X_iso = X_train[~iso_outliers]
y_iso = y_train[~iso_outliers]

print(y_train.value_counts())
print(y_iso.value_counts())
#1807 outliers(5% of the data) were removed. The class distribution remained relatively consistent and balanced.

Num of outliers = 1807
went_on_backorder
0    27103
1     9034
Name: count, dtype: int64
went_on_backorder
0    25750
1     8580
Name: count, dtype: int64


In [82]:
# ----------------------------------
pipe1 = Pipeline([ 
    ('scale', StandardScaler()),
    ('pca', PCA()),
    ('logreg', LogisticRegression(class_weight='balanced', max_iter = 3000, random_state=42))
]) 


In [83]:
#hyperparameter tuning
param_grid1 = [{'pca__n_components': [0.7, 0.8, 0.85, 0.9, 0.95],
              'logreg__C': [1e-2, 1e-1, 1, 1e1]}]

In [84]:
#Training
start = time()

model1 = GridSearchCV(pipe1, param_grid1, cv=5, n_jobs=1, scoring='f1')
model1.fit(X_iso, y_iso)

stop = time()


In [85]:
# ----------------------------------
# show tuning results: display scores, best model's parameters, classification report and confusion matrix for TRAINING data
print("Best F1:", model1.best_score_)
predicted_y1 = model1.predict(X_train)
print(classification_report(y_train, predicted_y1))
pd.DataFrame(confusion_matrix(y_train, predicted_y1))

Best F1: 0.6345651159824022
              precision    recall  f1-score   support

           0       0.95      0.70      0.81     27103
           1       0.50      0.90      0.64      9034

    accuracy                           0.75     36137
   macro avg       0.73      0.80      0.72     36137
weighted avg       0.84      0.75      0.76     36137



,0,1
0,18895,8208
1,896,8138


**Original analysis (see audit for qualifications)**

The model achieved an F1 score of 0.63 which is a good balance  between precision and recall. For the PCA hyperparameters, I tested different variance thresholds. I stared with a more coarse and wide thresholds then tested on the current one which helped improve the model. I tried adding L1 and L2 learning penalties to control regularization strength which didn't really helped with improving the resutls.

Overall, 0.89 recall value is very strong, however, this comes at the cost of lower precision (0.50) meaning a significant number of non-backorder products are incorrectly flagged. This is also reflected in the confusion matrix, the model prioritizes minimizing missed backorders over reducing false positives.

---


### Your 2nd pipeline
  * Anomaly detection
  * Dimensionality reduction
  * Model training/validation

In [88]:
# ----------------------------------

# Second pipeline: LocalOutlierFactor -> SelectKBest -> RandomForest

lo_factor = LocalOutlierFactor(n_neighbors = 150,contamination=0.05,)
lof_outliers = lo_factor.fit_predict(X_train) == -1
print(f"Num of outliers = {np.sum(lof_outliers)}")

X_lof = X_train[~lof_outliers]
y_lof = y_train[~lof_outliers]

print(y_train.value_counts())
print(y_lof.value_counts())

Num of outliers = 1807
went_on_backorder
0    27103
1     9034
Name: count, dtype: int64
went_on_backorder
0    25611
1     8719
Name: count, dtype: int64


In [91]:
# ----------------------------------
pipe2  = Pipeline([
    ('select', SelectKBest(score_func=mutual_info_classif)),
    ('rf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

In [92]:
param_grid2 = {
    'select__k': [5, 10, 15, 20],
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [5, 10, None],
    'rf__min_samples_leaf': [5, 10],
    'rf__max_features': ['sqrt']

}

In [94]:
start = time()

model2 = GridSearchCV(pipe2, param_grid2, cv=5, n_jobs=-1, scoring='f1')
model2.fit(X_lof, y_lof)

stop = time()


In [95]:
# ----------------------------------

# show tuning results: display scores, best model's parameters, classification report and confusion matrix for TRAINING data
print("Best F1:", model2.best_score_)
predicted_y2 = model2.predict(X_train)
print(classification_report(y_train, predicted_y2))
pd.DataFrame(confusion_matrix(y_train, predicted_y2))


Best F1: 0.8234206482303978
              precision    recall  f1-score   support

           0       0.98      0.93      0.96     27103
           1       0.82      0.95      0.88      9034

    accuracy                           0.94     36137
   macro avg       0.90      0.94      0.92     36137
weighted avg       0.94      0.94      0.94     36137



,0,1
0,25259,1844
1,441,8593


**Original analysis (see audit for qualifications)**

This pipeline significantly improved performance. The confusion matrix shows a large reduction in false postives. overall the model appears well-balanced and does not show clear signs of overfitting or underfitting. For Random Forest, I tuned different numbers of trees (n_estimators) and tree depth (max_depth) at first then later I added minimum leaf sizes (min_sample_leaf) which helped to improve the model and capture more complex patterns in the data.

---

### Your 3rd pipeline
  * Anomaly detection
  * Dimensionality reduction
  * Model training/validation

In [96]:
# ----------------------------------

#EllipticEnvelope -> FactorAnalysis -> GradientBoostingClassifier
ele = EllipticEnvelope(contamination=0.05, random_state=42)
ele_outliers = ele.fit_predict(X_train) == -1
print(f"Num of outliers = {np.sum(ele_outliers)}")

X_ele = X_train[~ele_outliers]
y_ele = y_train[~ele_outliers]

print(y_train.value_counts())
print(y_ele.value_counts())


Num of outliers = 1807
went_on_backorder
0    27103
1     9034
Name: count, dtype: int64
went_on_backorder
0    25610
1     8720
Name: count, dtype: int64


In [97]:
# ----------------------------------
pipe3  = Pipeline([
    ('fa', FactorAnalysis()),
    ('gb', GradientBoostingClassifier(random_state=42))
])

In [98]:
param_grid3 = {
    'fa__n_components': [5,10],
    'gb__n_estimators': [100],
    'gb__learning_rate': [0.05, 0.1],
    'gb__max_depth': [2,3,5],
    'gb__min_samples_leaf': [3,5,10]
}

In [100]:
start = time()

model3 = GridSearchCV(pipe3, param_grid3, cv=3, n_jobs=-1, scoring='f1')
model3.fit(X_ele, y_ele)

stop = time()

In [101]:
# ----------------------------------

# show tuning results: display scores, best model's parameters, classification report and confusion matrix for TRAINING data
print("Best F1:", model3.best_score_)
predicted_y3 = model3.predict(X_train)
print(classification_report(y_train, predicted_y3))
pd.DataFrame(confusion_matrix(y_train, predicted_y3))


Best F1: 0.7732656233824217
              precision    recall  f1-score   support

           0       0.93      0.94      0.94     27103
           1       0.82      0.80      0.81      9034

    accuracy                           0.91     36137
   macro avg       0.88      0.87      0.87     36137
weighted avg       0.91      0.91      0.91     36137



,0,1
0,25510,1593
1,1782,7252


**Original analysis (see audit for qualifications)**

pipeline3 has a moderate performance with an F1-score of ~0.76 and it also maintained a reasonable balance between precision and recall. Compared to the other pipelines, it has lower recall resulting in more missed backorder cases. I tuned the Gradient Boosting parameters which helped balance learning speed and model complexity. At first, I stared off with just n_estimators and learning_rate and the model performed well with 0.72 F1 score, which was good. Adding the other hyprerparameters and tweaking those values helped the model to do better with a cost of longer training time. Overall, Pipeline 3 preformed better than pipeline 1 but not as well as pipeline 2.

---

## Compare these three pipelines and discuss your findings

Do NOT fit here again, just use the predict function with the test data and report which of the three pipelines perform the best. It is up to you what you deem "best". 

In [106]:
#Pipeline1
print("Pipeline1 results")
predicted_y1_test = model1.predict(X_test)
print(classification_report(y_test, predicted_y1_test))
pd.DataFrame(confusion_matrix(y_test, predicted_y1_test))

Pipeline1 results
              precision    recall  f1-score   support

           0       0.96      0.70      0.81      6776
           1       0.50      0.91      0.65      2259

    accuracy                           0.75      9035
   macro avg       0.73      0.80      0.73      9035
weighted avg       0.84      0.75      0.77      9035



,0,1
0,4733,2043
1,206,2053


In [107]:
#Pipeline2
print("Pipeline2 results")
predicted_y2_test = model2.predict(X_test)
print(classification_report(y_test, predicted_y2_test))
pd.DataFrame(confusion_matrix(y_test, predicted_y2_test))

Pipeline2 results
              precision    recall  f1-score   support

           0       0.96      0.91      0.94      6776
           1       0.78      0.89      0.83      2259

    accuracy                           0.91      9035
   macro avg       0.87      0.90      0.88      9035
weighted avg       0.92      0.91      0.91      9035



,0,1
0,6194,582
1,241,2018


In [109]:
#Pipeline3
print("Pipeline3 results")
predicted_y3_test = model3.predict(X_test)
print(classification_report(y_test, predicted_y3_test))
pd.DataFrame(confusion_matrix(y_test, predicted_y3_test))

Pipeline3 results
              precision    recall  f1-score   support

           0       0.92      0.93      0.92      6776
           1       0.77      0.76      0.77      2259

    accuracy                           0.88      9035
   macro avg       0.85      0.84      0.85      9035
weighted avg       0.88      0.88      0.88      9035



,0,1
0,6270,506
1,539,1720


**Original analysis (see audit for qualifications)**

Final Comparison of the three pipelines
Pipeline 1
- Very high recall: 0.91
- Very low precision: 0.50
- Too many false positives: 2043

- It is good at detecting backorders but it is unreliable since it has too many false positives. Seems like the model became aggressive about predicting the minority class. It cathes most of the backorders(91% recall) but it also raises many unnecessary alarms(50% precision) which could mean the model is underfitting.

Pipeline 2
- It has the best balance overall
- Good precision: 0.78
- Strong recall: 0.89
- Best F1 score: 0.83
- Lowest overall mistakes when it comes to missing backorders and false positives.

Pipeline 3
- Better precision than pipeline 1: 0.77
- Lower recall than pipeline 2: 0.76
- Missed more actual backorders and weaker overall than pipeline 2

Overall: 
- Pipeline 2 performed the best because:
    - It reduced false positives dramatically
    - Has high recall and the best F1 score compared to the other two
    - Generlized well on unseen data

---

### Pickle the required pipeline/models and save them to use in Part 3.

Save your best pipeline and the anomaly detection model. Do NOT save the grid object, save the **best_estimator_** of the grid object. 

In [114]:

#balanced sample
with open('sample.pkl', 'wb') as f:
    pickle.dump((X_lof, y_lof), f)
#anomaly detection model
with open('lof.pkl','wb') as f:
    pickle.dump(lo_factor, f)
#Best pipeline
with open('final_pipeline.pkl', 'wb') as f:
    pickle.dump(model2.best_estimator_, f)

